## Tratamento de Dados

- Removida a coluna `duration`, pois causa vazamento temporal (só é conhecida após a ligação acontecer).
- Valores "unknown" em colunas categóricas foram mantidos como categoria própria, 
  em vez de imputados, por representarem uma resposta legítima do cliente.
- Variáveis categóricas transformadas via one-hot encoding (10 colunas → 53 colunas).
- Target `y` convertido para binário (0 = não converteu, 1 = converteu).
- Dataset final: 41.188 clientes, taxa de conversão de referência = 11.3%.

In [1]:
import pandas as pd

df = pd.read_csv("../data/bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [2]:
df.info()
df.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

In [5]:
df['y'].value_counts()
df['y'].value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [6]:
df = df.drop(columns=['duration'])
df.shape

KeyError: "['duration'] not found in axis"

In [7]:
colunas_categoricas = df.select_dtypes(include='object').columns.tolist()
for col in colunas_categoricas:
    print(col, '->', df[col].unique())

job -> ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital -> ['married' 'single' 'divorced' 'unknown']
education -> ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
default -> ['no' 'unknown' 'yes']
housing -> ['no' 'yes' 'unknown']
loan -> ['no' 'yes' 'unknown']
contact -> ['telephone' 'cellular']
month -> ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week -> ['mon' 'tue' 'wed' 'thu' 'fri']
poutcome -> ['nonexistent' 'failure' 'success']
y -> ['no' 'yes']


In [8]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

In [ ]:
colunas_numericas = ['age', 'campaign', 'pdays', 'previous',
                      'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
                      'euribor3m', 'nr.employed']

colunas_categoricas = ['job', 'marital', 'education', 'default', 'housing',
                        'loan', 'contact', 'month', 'day_of_week', 'poutcome']

print("Numéricas:", len(colunas_numericas))
print("Categóricas:", len(colunas_categoricas))

Numéricas: 9
Categóricas: 10


In [10]:
df_encoded = pd.get_dummies(df, columns=colunas_categoricas, drop_first=True)
df_encoded.shape

(41188, 53)

In [11]:
df_encoded['y'] = df_encoded['y'].map({'no': 0, 'yes': 1})
df_encoded['y'].value_counts()

y
0    36548
1     4640
Name: count, dtype: int64

In [12]:
df_encoded.to_csv("../data/processed.csv", index=False)

In [13]:
taxa_conversao_baseline = df_encoded['y'].mean()
print(f"Taxa de conversão do Baseline: {taxa_conversao_baseline:.4f} ({taxa_conversao_baseline*100:.2f}%)")

Taxa de conversão do Baseline: 0.1127 (11.27%)


## Baseline

O baseline representa a política atual: ofertar para todos os clientes elegíveis, 
sem nenhuma personalização ou estratégia adaptativa. A taxa de conversão observada 
no histórico é de aproximadamente 11.3% — esse é o número que a política adaptativa 
(bandit) precisa superar para justificar sua adoção.

In [21]:
import numpy as np
np.random.seed(42)

segmentos = df['poutcome'].unique().tolist()
dados_por_segmento = {s: df_encoded[df['poutcome'] == s]['y'].values for s in segmentos}

for s in segmentos:
    print(s, "taxa média:", dados_por_segmento[s].mean(), "| n:", len(dados_por_segmento[s]))

nonexistent taxa média: 0.08832213255349661 | n: 35563
failure taxa média: 0.1422859830667921 | n: 4252
success taxa média: 0.651128914785142 | n: 1373


In [23]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.start_run()
mlflow.log_param("algoritmo", "thompson_sampling_por_segmento")
mlflow.log_param("segmentos", segmentos)
mlflow.log_param("budget", budget)
mlflow.log_metric("taxa_conversao_baseline", taxa_conversao_baseline)
mlflow.log_metric("taxa_conversao_bandit", taxa_conversao_bandit)
mlflow.end_run()

In [24]:
golden_set = df.sample(5, random_state=1)
for i, row in golden_set.iterrows():
    seg = row['poutcome']
    amostras = {s: np.random.beta(sucessos[s], falhas[s]) for s in segmentos}
    melhor_segmento = max(amostras, key=amostras.get)
    decisao = "OFERTAR" if seg == melhor_segmento else "NÃO PRIORIZAR"
    print(f"Cliente {i}: idade={row['age']}, segmento={seg}, decisão={decisao}")

Cliente 35577: idade=32, segmento=nonexistent, decisão=NÃO PRIORIZAR
Cliente 13950: idade=33, segmento=nonexistent, decisão=NÃO PRIORIZAR
Cliente 29451: idade=25, segmento=nonexistent, decisão=NÃO PRIORIZAR
Cliente 32295: idade=34, segmento=nonexistent, decisão=NÃO PRIORIZAR
Cliente 27477: idade=53, segmento=nonexistent, decisão=NÃO PRIORIZAR
